# Function 1 — from observations to a first query

Ten observations of a function with two inputs, `x1` and `x2`, each in `[0, 1)`.
Larger outputs are better. We explore the supplied data, then use its sampling
gaps to select a point for the first new query.

**Taper-waper: start by reducing ignorance, finish by exploiting what we have learned.**

In [ ]:
from pathlib import Path
import sys

stage = next(p / "stage-2-bbo" for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "stage-2-bbo/src").is_dir())
sys.path.insert(0, str(stage / "src"))
from initial_exploration import load_initial_data, preview, plot_ranked_locations, plot_sampling_gaps, plot_ordered_outputs

data = load_initial_data(stage, function_id=1)
figures = stage.parent / ".runtime/function-1-initial-analysis"
from initial_exploration import maximin_candidates_2d, maximin_comparison, plot_maximin_choice
from initial_exploration import coverage_value_comparison
from coverage_optimisation import optimise_average_coverage
from initial_exploration import average_coverage_summary, plot_average_coverage

## A quick look at the values

All ten observations, in their supplied order. Throughout this notebook,
**rank 1 is the highest output and rank 10 is the lowest**; ranks are not row IDs.

In [ ]:
preview(data, rows=len(data))

## Where the observations sit

Each marker shows one observed location. Its number and colour give its output
rank, so the ordering remains visible despite the enormous differences in scale.

In [ ]:
plot_ranked_locations(data, save_to=figures / "ranked-locations.png")

The best observed point is near **(0.731, 0.733)**; the worst is near
**(0.650, 0.682)**. They are also the closest pair of observations, about
0.096 units apart. That contrast is worth noticing, but does not establish the
shape of the function between them.

Ranks show order, not the size of the differences. In particular, the highest
output is only about **7.71 × 10⁻¹⁶**; being ranked first does not establish that
we have found a substantial positive response.

## Sampling gaps

Brighter areas are farther from the nearest supplied observation. The marker
numbers retain the same output ranks as above. The colour describes **gaps in
sampling**, not predicted output or model uncertainty.

In [ ]:
plot_sampling_gaps(data, save_to=figures / "sampling-gaps.png")

A broad central gap lies near **(0.42, 0.46)**, and several boundary regions are
also sparsely sampled. Both coordinates individually span roughly 0.08–0.88,
but that does not mean their combinations have been well explored. A narrow
feature could still lie between the observations.

## How large are the outputs?

Both panels run from the highest output on the left to the lowest on the right.
Blue circles represent positive outputs; orange triangles represent negative
outputs.

In [ ]:
plot_ordered_outputs(data, save_to=figures / "ordered-outputs.png")

There are **seven positive outputs and three negative outputs**, with no exact
zeros. One negative value, **−0.003606…**, dominates the original-scale plot.
The other nine magnitudes are all at most **7.71 × 10⁻¹⁶**.

The logarithmic view reveals a spread of about **121 orders of magnitude**.
Each step of +1 on its vertical axis means ten times the magnitude. The three
negative outputs appear at ranks 8, 9 and 10; a higher position in this panel
means a larger magnitude, not necessarily a better result.

### With the largest-magnitude output excluded

Rank 10 (approximately -0.003606) is omitted from these charts only. The raw
axis is rescaled; the ranks and logarithmic axis keep the full-data reference.

In [ ]:
plot_ordered_outputs(data, exclude_rank=10, save_to=figures / "ordered-outputs-without-outlier.png")

The largest positive output now dominates the raw scale. The other eight
values still appear close to zero on that scale; the logarithmic view separates
their magnitudes and retains the two remaining negative outputs.

These tiny values are present in the supplied data. The observations alone do
not establish whether differences at that scale will help locate a larger
positive response. The central sampling gap and nearby best/worst pair remain
the clearest spatial features of this starting snapshot.

## A first coverage rule: maximin

The highest observed output is a relative best. These observations do not
establish a useful target value or a noise level, so they give us little basis
for concentrating the first query around rank 1. We start with **coverage**:
the early part of the taper addresses large spatial blind spots before
committing more queries to an apparent optimum.

**Sequential maximin sampling** means **maximise the minimum distance**:

1. At each candidate location, measure the distance to the nearest existing observation.
2. Choose the candidate with the largest such distance.

This is the score already displayed by the sampling-gap heatmap. It uses the
input locations alone; the output values do not enter the calculation.
All ten locations count, including rank 10, which was excluded only from the
additional output-scale charts. The method is described in
[Sequential maximin design](https://bookdown.org/rbg/surrogates/chap4.html#sequential-maximin-design).

Both inputs share the unit interval, and we give them equal weight in ordinary
straight-line distance. We include the edges and corners rather than favouring
the interior. With six decimal places and values strictly below 1, the largest
allowed coordinate is **0.999999**.

In [ ]:
candidates = maximin_candidates_2d(data)
maximin_comparison(candidates)

The central gap is a plausible candidate by eye. Applying the same rule across
the domain selects the **top-left corner**, whose nearest observation is about
**0.398** away, compared with **0.316** for the central alternative.

The calculation checks the interior gaps, domain edges and corners. The selected
corner also attains the maximum before coordinates are rounded to six decimal
places; this result is not just the brightest cell of a coarse plotting grid.

In [ ]:
plot_maximin_choice(data, candidates, save_to=figures / "maximin-choice.png")

The star is the selected point. The dashed line connects it to its nearest
existing observation, labelled rank 5. The diamond marks the central alternative.
The heatmap still represents distance, not predicted output.

## Does the corner make the best use of a query?

An observation at the corner may help cover less usable space than one slightly
inward. If its influence were a fixed circular neighbourhood, only a quarter
of that circle would lie inside the square at an exact corner. There is no
known radius of influence here, so this is an intuition, not a measured loss
of three quarters of the information.

Maximin asks how isolated the new point is. A different coverage question is:
**how much closer does the new point bring the rest of the usable space to a
sample?**

For that comparison, take a dense, evenly spaced grid inside the square and
measure each location's distance to its nearest observation before and after
adding a candidate. Average those distances, giving every part of the square
equal weight. This uses geometry only; it does not evaluate the unknown function.


In [ ]:
coverage_value_comparison(data, candidates)

The inset point **(0.07, 0.93)** improves the average distance more than the
corner: **5.64% versus 3.37%**. It also brings more of the square closer to a
sample. The central gap gives the largest average improvement of these three,
at **7.38%**; it has not been established as the global optimum of that criterion.

The two criteria answer different questions. The corner wins on isolation.
The inset and central points improve average coverage more. Neither score
predicts a high output or quantifies actual information gained about the function.

## Optimising average coverage

There is no need to stop at three hand-picked candidates. Their comparison gives
us an objective we can optimise:

**Minimise the average distance from a location in the square to its nearest
sample, after adding the new point.**

Every part of the square receives equal weight. All ten input locations count,
and the supplied outputs do not affect the score. We can evaluate this objective
locally as often as needed, without spending any capstone evaluations.

The calculation integrates distance over the regions belonging to each nearest
sample, avoiding a grid approximation to the average. A
[differential-evolution search](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.differential_evolution.html)
explores the domain, followed by local refinement.

In [ ]:
coverage_optimum = optimise_average_coverage(data)
average_coverage_summary(coverage_optimum)

The numerical recommendation is **(0.468429, 0.435440)**. It reduces mean
sampling distance by **7.54%**, compared with **7.38%** at the central-gap point,
**5.64%** at the inset corner and **3.37%** at the maximin corner.

The gain over the central alternative is modest, but the location now follows
the chosen criterion rather than an arbitrary inset or visual estimate.

In [ ]:
plot_average_coverage(data, coverage_optimum, save_to=figures / "average-coverage-choice.png")

Here the colour scores **where to put the new point**: brighter locations give
a larger reduction in average sampling distance. It is not a prediction of the
unknown function. Existing samples retain their output-rank labels.

Two independently seeded global searches and refinement from other candidate
regions agree on the same location. Independent numerical integration checks
support the geometric calculation. This is a well-checked numerical result,
not a formal proof of the global optimum; the six decimal places make the
proposal reproducible, not certain to that precision.

## Our first query under average coverage

**Proposed coordinates: `(0.468429, 0.435440)`**

Portal format:

```text
0.468429-0.435440
```

**Status: proposed; not submitted.**

The choice assumes that improving average coverage of the whole square is useful
for this first experiment. It does not predict a large positive output or
guarantee that a narrow peak will be found.

Once the output is available, the new observation will update the ranking and
coverage map. The remaining 12 choices can respond to that evidence. A more
informative response can shift attention towards a promising region; another
tiny output would leave the location of any strong response unresolved.

## The taper in Function 1

Across the 13 evaluation rounds, the emphasis shifts from exploration towards
fine tuning. Early queries reduce consequential blind spots and test competing
explanations. Middle rounds concentrate on promising regions while checking
alternatives. Later rounds refine the strongest candidates and assess whether
apparent gains are credible in the presence of noise.

The taper responds to evidence: an unexpected result can justify renewed
exploration, and each function can progress at a different pace. Coverage is one
way to reduce ignorance; it does not measure the probability of missing the
optimum. The recurring question is **what uncertainty does this query address,
and why is resolving it worth one of the remaining attempts?**

Here the first uncertainty is where a useful response might lie. The log-magnitude
view makes the supplied outputs readable, but does not establish that their
tiny differences identify a promising region. Optimising average sampling
distance gives the first query a clear geometric purpose. The change from
maximin to average coverage also makes the intended benefit explicit: bringing
more of the usable domain closer to an observation.

As new evidence arrives, the emphasis can move towards fitting the response and
refining promising locations. Later queries will need to justify that shift;
elapsed rounds alone do not make the current best observation reliable.

## What changes as dimensions increase?

This two-input case is our baseline: one two-dimensional plot shows every input
coordinate. With three inputs there are **3 coordinate pairs**; with eight there
are **28**. Those plots show projections. Points that look close in one projection
can be far apart in the full input space, so pairwise maps cannot establish full
coverage.

Even a toy grid with five positions per input grows from **25 combinations in
two dimensions** to **125 in three** and **390,625 in eight**. That is an
illustration of the growth, not a proposed sampling plan. As dimensions increase,
our comparison will track how the reason for a query changes: filling a gap,
testing an interaction between inputs, distinguishing fitted explanations, or
refining a promising region. Any fitted emphasis on particular inputs will need
evidence; higher dimension alone does not justify ignoring the others.

The later functions also have different behaviours and numbers of starting
observations. Differences in our approach cannot automatically be attributed to
dimension. The pattern to watch is **how the uncertainty worth spending a query
on changes**, while the taper remains our organising principle.